In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train = pd.read_csv(r'/content/Modelling_ready_train_data.csv').iloc[:,1:]
val = pd.read_csv(r'/content/Modelling_ready_validation_data.csv').iloc[:,1:]

In [ ]:
# Adding <START> / <END> tokens To every output sentence for the training
# data so that my model knows when a output starts and when it ends

train['Correction'] = '<START> ' + train['Correction'] + ' <END>'

In [ ]:
# Tokeinzing

In [ ]:
# Training Training input each sentence length
print(' Input corpus length distribution')
print(train.Sentence.str.split().str.len().describe())

# Training Training input each sentence length
print(' Output corpus length distribution')
print(train.Correction.str.split().str.len().describe())



 Input corpus length distribution
count    2500.000000
mean       18.036800
std         9.843363
min         1.000000
25%        11.000000
50%        16.000000
75%        23.000000
max        76.000000
Name: Sentence, dtype: float64
 Output corpus length distribution
count    2500.000000
mean       19.978400
std         9.670416
min         3.000000
25%        13.000000
50%        18.000000
75%        25.000000
max        77.000000
Name: Correction, dtype: float64


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Handling Punctuation in Grammatical Error Correction

- **Keep punctuation marks** in both input and output corpora.  
- **Do not strip punctuation** during tokenization — punctuation errors are part of the correction task.  
- **Normalize spacing** around punctuation for consistency (e.g., `"word ."` → `"word."`).  
- **Preserve sentence boundaries** — punctuation helps the model understand structure.  
- **Tokenizer setup**:  
  - Use `filters=''` so punctuation is not removed.  
  - Ensure `<start>`, `<end>`, and `<unk>` tokens remain intact.  
- **Training benefit**:  
  - Model learns to correct missing, misplaced, or mis‑spaced punctuation.  
  - Evaluation aligns with reference corrections that include punctuation.


In [ ]:
# corpus
# Input corpus
input_corpus  = train['Sentence'].tolist()
output_corpus  = train['Correction'].tolist()

# tokenization
tokenizer_input = Tokenizer(oov_token = '<OOV>' ,filters='', lower=False)
tokenizer_output = Tokenizer(oov_token = '<OOV>' , filters='', lower=False)

In [ ]:
# Tokenizer fitted
tokenizer_input.fit_on_texts(input_corpus)
tokenizer_output.fit_on_texts(output_corpus)

# Now the tokenizer is fitted on training data it will be used for evaluation and test data

In [ ]:
# vocabulary
input_vocab = tokenizer_input.word_index
output_vocab = tokenizer_output.word_index

# Creating the input and output feeding data for model.fit

In [ ]:
final_feeding_input = tokenizer_input.texts_to_sequences(input_corpus)
final_feeding_output = tokenizer_output.texts_to_sequences(output_corpus)



# Apply PADDING
input_lengths = train['Sentence'].str.split().str.len()
output_lengths = train['Correction'].str.split().str.len()

MAX_LEN_INPUT = int(np.percentile(input_lengths, 95))
MAX_LEN_OUTPUT = int(np.percentile(output_lengths, 95))

print(MAX_LEN_INPUT, MAX_LEN_OUTPUT)

final_feeding_input = pad_sequences(final_feeding_input, maxlen=MAX_LEN_INPUT, padding='post')
final_feeding_output = pad_sequences(final_feeding_output, maxlen=MAX_LEN_OUTPUT, padding='post')

import joblib

joblib.dump(tokenizer_input, 'tokenizer_input.pkl')
joblib.dump(tokenizer_output, 'tokenizer_output.pkl')

# Now the model i and for final evaluation of the test data plus
# validation data we will use this tokenizer for converting that data into model ready input data

35 41


# Loading the word two week model for extracting its vector embeddings off my vocab words for both input output

In [ ]:
# Scenes we are using the embeddings of what to work on pre trade models
# we don't need convert the vocab into integer encoding and padding

# We can directly call the word to model and extract the embeddings of my particular vocab
!pip install gensim
import gensim.downloader as api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.7 MB/s eta 0:00:00


In [ ]:
word2vec = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


# Workflow for Building Embedding Matrix

- **Step 1: Initialize matrix**
  - Create a zero matrix of shape `(vocab_size + 1, 300)` for both input and output vocabularies.
  - The extra `+1` ensures index `0` exists, which is reserved for padding when using `pad_sequences`.

- **Step 2: Fill with pretrained embeddings**
  - For words present in the pretrained model (Word2Vec/FastText/GloVe), copy their 300‑dimensional vectors into the corresponding row.
  - This gives known words meaningful starting representations.

- **Step 3: Handle missing words**
  - For words not found in the pretrained embeddings (including special tokens like `<start>`, `<end>`, `<OOV>`):
    - Initialize their rows with random values drawn from a normal distribution.
    - Keep these embeddings trainable so the model can learn them during training.

- **Step 4: Padding row**
  - Row `0` remains all zeros.
  - This row is automatically used when sequences are padded with `0` by Keras/TensorFlow.

- **Step 5: Finalize matrix**
  - Result is a hybrid embedding matrix:
    - Pretrained vectors for known words.
    - Random normal vectors for unknown/special tokens.
    - Zero row for padding.

- **Step 6: Train inside the model**
  - Use the matrix to initialize your `Embedding` layer with `trainable=True`.
  - The model fine‑tunes pretrained vectors and learns embeddings for special/unknown tokens during training.


In [ ]:

# Input embedding matrix
input_emb = np.zeros((len(input_vocab) + 1, 300))
for token, idx in input_vocab.items():
    if token in word2vec.key_to_index:
        input_emb[idx] = word2vec[token]
    else:
        np.random.seed(idx + 5)
        input_emb[idx] = np.random.normal(scale=0.6, size=(300,))

# Output embedding matrix
output_emb = np.zeros((len(output_vocab) + 1, 300))
for token, idx in output_vocab.items():
    if token in word2vec.key_to_index:
        output_emb[idx] = word2vec[token]
    else:
        np.random.seed(idx + 5)
        output_emb[idx] = np.random.normal(scale=0.6, size=(300,))


In [ ]:
word2vec_input_emb = input_emb.copy()
word2vec_output_emb = output_emb.copy()

# Saving and Reusing Embedding Matrices

- **Build embedding matrices**  
  - Create `(vocab_size, 300)` matrices for input and output vocabularies.  
  - Fill with pretrained vectors where available, random normal for special/unknown tokens.

- **Save matrices locally**  
  - Use NumPy for efficient storage:  
    ```python
    np.save("input_embedding_matrix.npy", input_embedding_matrix)
    np.save("output_embedding_matrix.npy", output_embedding_matrix)
    ```

- **Reload when needed**  
  - Quickly load without downloading Word2Vec/GloVe again:  
    ```python
    input_embedding_matrix = np.load("input_embedding_matrix.npy")
    output_embedding_matrix = np.load("output_embedding_matrix.npy")
    ```

- **Use in model**  
  - Pass loaded matrices into Keras `Embedding` layers with `trainable=True` to fine‑tune during training.


In [ ]:
np.save("word2vec_input_embedding_matrix.npy", word2vec_input_emb)
np.save("word2vec_output_embedding_matrix.npy", word2vec_output_emb)

# Using fast text model for extracting embedding vectors

In [ ]:
fasttext_model = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:

# Input embedding matrix

input_emb = np.zeros((len(input_vocab) + 1, 300))
for token, idx in input_vocab.items():
    if token in fasttext_model.key_to_index:
        input_emb[idx] = fasttext_model[token]
    else:
        np.random.seed(idx + 5)
        input_emb[idx] = np.random.normal(scale=0.6, size=(300,))

# Output embedding matrix
output_emb = np.zeros((len(output_vocab) + 1, 300))
for token, idx in output_vocab.items():
    if token in fasttext_model.key_to_index:
        output_emb[idx] = fasttext_model[token]
    else:
        np.random.seed(idx + 5)
        output_emb[idx] = np.random.normal(scale=0.6, size=(300,))


In [ ]:
fasttext_input_emb = input_emb.copy()
fasttext_output_emb = output_emb.copy()

In [ ]:
np.save("fasttext_input_embedding_matrix.npy", fasttext_input_emb)
np.save("fasttext_output_embedding_matrix.npy", fasttext_output_emb)